## Prepare Environment

In [ ]:
!pip install --upgrade pip
!pip install --upgrade "datasets[audio]" transformers accelerate evaluate jiwer tensorboard gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 131.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 60.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.3 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.1
    Uninstalling transformers-5.20.1:
      Successfully uninstall

## Load Dataset

In [ ]:
!git clone https://github.com/stanford-policylab/asr-disparities.git

Cloning into 'asr-disparities'...
remote: Enumerating objects: 6195, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 6195 (delta 4), reused 0 (delta 0), pack-reused 6186 (from 1)
Receiving objects: 100% (6195/6195), 848.74 MiB | 15.34 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Updating files: 100% (6163/6163), done.


In [ ]:
!ls asr-disparities/input/

CORAAL_audio  CORAAL_transcripts.csv  DDM.csv  VOC_WER.csv


In [ ]:
from glob import glob
# coraal_audio = sorted(glob("asr-disparities/input/CORAAL_audio/*/*.mp3"))
coraal_audio = sorted(glob("asr-disparities/input/CORAAL_audio/dcb/*.mp3"))

In [ ]:
import pandas as pd
gt_csv = "asr-disparities/input/CORAAL_transcripts.csv" #csv with ground truth transcripts
gt_df = pd.read_csv(gt_csv)
gt_df = gt_df.sort_values(by='segment_filename')
gt_df.head()

,segment_filename,basefile,age,gender,source,duration,race_ethnicity,content,google_transcription,ibm_transcription,amazon_transcription,msft_transcription,apple_transcription
26,DCB_se1_ag2_f_01_1_1002411_1038909.wav,DCB_se1_ag2_f_01_1,28,Female,DCB,36.4979,Black,and I f- I really felt like I was the smartest...,and I really felt like I was the smartest pers...,I really felt like I was the smartest person a...,enough. I've really felt like I was the smarte...,I really felt like I was the smartest person a...,And I've I've really felt like I was the smart...
27,DCB_se1_ag2_f_01_1_1058180_1106086.wav,DCB_se1_ag2_f_01_1,28,Female,DCB,47.9054,Black,It was just like that. Like books- It w- They ...,it was just like that like books and what they...,it was just like that like books it would they...,It was just like that. Like books And what The...,"It was just like that, like books at what they...",It was just like that like books it with they ...
28,DCB_se1_ag2_f_01_1_1137188_1154185.wav,DCB_se1_ag2_f_01_1,28,Female,DCB,16.9967,Black,Thir- Third World's Third Street. It's really-...,third world 3rd Street it's really a berry far...,there were a thirsty it's really a berry farms...,they're They're world's third Street. It's rea...,3rd World 3rd St. Is really very farms had ver...,Their world Thursday is really a berry farms M...
29,DCB_se1_ag2_f_01_1_1191138_1197065.wav,DCB_se1_ag2_f_01_1,28,Female,DCB,5.9275,Black,It was so many girl gangs at Eastern. And I wa...,it was so funny girl games,and I was cool with all of them to I didn't I ...,It was so many girl gangs. Ace And I was cool ...,"It was so many girl games, please. And that wa...",I didn't click it
30,DCB_se1_ag2_f_01_1_1201294_1216302.wav,DCB_se1_ag2_f_01_1,28,Female,DCB,15.0083,Black,"like now, that's how I know so many people. It...",like now that's how I know so many people it w...,that's how I know so many people there was mos...,like male. That's how I know so many people. I...,Live mail that's how I know so many people. It...,Like now that's how I know so many people ther...


Make sure to only get audio files with their transcript present in the test set

In [ ]:
base_audio_name =[item.split("/")[-1].split(".")[0] for item in coraal_audio]
base_transcript = [item.replace(".wav","") for item in gt_df['segment_filename'].tolist()]
use_set = sorted(list(set(base_audio_name) & set(base_transcript)))

coraal_audio_set = []
for item in use_set:
    get_file = glob(f"asr-disparities/input/CORAAL_audio/*/{item}.mp3")
    coraal_audio_set.append(get_file[0])



In [ ]:
import os
import re

gt_trans = []
file_names = []

# Create a dictionary for efficient lookup: base_filename_without_ext -> content
# This assumes segment_filename in gt_df ends with '.wav' and coraal_audio elements end with '.mp3'
gt_content_map = {
    os.path.splitext(row['segment_filename'])[0]: row['content']
    for index, row in gt_df.iterrows()
}

for audio_path in coraal_audio_set:
    # Extract the base filename without extension from the audio path
    base_audio_name = os.path.splitext(os.path.basename(audio_path))[0]
    file_names.append(os.path.basename(audio_path))

    # Look up the transcript using the base filename.
    # Use an empty string if no matching transcript is found for this audio file.
    transcript = gt_content_map.get(base_audio_name, '')

    # Remove any annotation notes in angle brackets (e.g., <ts>, <noise>)
    transcript = re.sub(r'<.*?>', '', transcript).strip()

    gt_trans.append(transcript)

# Display the first transcript as requested by the original cell
print(gt_trans[0])

and I f- I really felt like I was the smartest person at school because they didn't learn half of what I already knew yet.  And w- The reason I went, I uh- had /RD-SCHOOL-3/ which is Uptown- I had to go to /RD-SCHOOL-3/ because a credit of mine in Maryland didn't count in DC so I had to take n- a night school class. So but, yeah that's why I said that. And when I got to Eastern, it was drinking. It was smoking. I could not believe where I was- I was like, oh my God. Ma, you need to get me out of this school. But I got used to it.  I got used to it.


In [ ]:
df_audio_files = pd.DataFrame({'file_name': file_names})
df_audio_files["transcription"] = gt_trans
print(f"DataFrame created with {len(file_names)} audio file paths.")

display(df_audio_files.head())
csv_file_path = "asr-disparities/input/CORAAL_audio/dcb/metadata.csv"
df_audio_files.to_csv(csv_file_path, index=False)
print(f"DataFrame successfully saved to {csv_file_path}")

DataFrame created with 2384 audio file paths.


,file_name,transcription
0,DCB_se1_ag2_f_01_1_1002411_1038909.mp3,and I f- I really felt like I was the smartest...
1,DCB_se1_ag2_f_01_1_1058180_1106086.mp3,It was just like that. Like books- It w- They ...
2,DCB_se1_ag2_f_01_1_1191138_1197065.mp3,It was so many girl gangs at Eastern. And I wa...
3,DCB_se1_ag2_f_01_1_1201294_1216302.mp3,"like now, that's how I know so many people. It..."
4,DCB_se1_ag2_f_01_1_1256071_1262822.mp3,I don't know. I was- I was never scared. Only ...


DataFrame successfully saved to asr-disparities/input/CORAAL_audio/dcb/metadata.csv


In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("asr-disparities/input/CORAAL_audio/dcb")
dataset = dataset['train']
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

Resolving data files:   0%|          | 0/3670 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

In [ ]:
train_dataset = train_dataset.map(
    prepare_dataset,
    num_proc=1,
)

eval_dataset = eval_dataset.map(
    prepare_dataset,
    num_proc=1,
)

In [ ]:
print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['audio', 'transcription'],
    num_rows: 2384
})
Dataset({
    features: ['audio', 'transcription'],
    num_rows: 2145
})
Dataset({
    features: ['audio', 'transcription'],
    num_rows: 239
})


## Load Model

In [ ]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [ ]:
MODEL_CHECKPOINT = "openai/whisper-medium"

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_CHECKPOINT, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_CHECKPOINT, task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

## Inference

In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

In [ ]:
predictions = []
references = []

for example in eval_dataset:
    result = asr(
        example["audio"],
        return_timestamps=True,
        generate_kwargs={
            "task": "transcribe",
        },
    )
    predictions.append(result["text"])
    references.append(example["transcription"])

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor

In [ ]:
baseline_df = pd.DataFrame({
    "reference": references,
    "zero_shot_prediction": predictions,
})
baseline_df.head()
baseline_df.to_csv("whisper_baseline.csv", index=False)

## Evaluate

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
zero_shot_wer = wer_metric.compute(
    predictions=predictions,
    references=references,
)
print(f"Zero-shot WER: {zero_shot_wer}")

Zero-shot WER: 0.3167157640305159


## Train

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-fine-tune",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=200,
    gradient_checkpointing=True,
    fp16=True,
    num_train_epochs=10,
    # eval_strategy="steps",
    eval_strategy="no",
    logging_strategy="epoch",
    save_total_limit=2,
    save_strategy="epoch",
    predict_with_generate=True,
    report_to=["tensorboard"],
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    # eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

In [ ]:
trainer.train()

Exception ignored in: <_io.BytesIO object at 0x7da043a18900>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7da0439989f0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7da0b6865c10>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7da0b68640e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-package

Step,Training Loss
135,0.980041
270,0.529155
405,0.387526
540,0.290805
675,0.209451
810,0.146860
945,0.102058
1080,0.070106
1215,0.050089
1350,0.038355


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1350, training_loss=0.2804445941359909, metrics={'train_runtime': 4959.9072, 'train_samples_per_second': 4.325, 'train_steps_per_second': 0.272, 'total_flos': 6.190156836864e+18, 'train_loss': 0.2804445941359909, 'epoch': 10.0})

In [ ]:
trainer.save_model("./whisper-small-fine-tune")

asr_finetuned = pipeline(
    "automatic-speech-recognition",
    model="./whisper-small-fine-tune",
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
predictions_finetuned = []
references_finetuned = []

for example in eval_dataset:
    result = asr_finetuned(
        example["audio"],
        return_timestamps=True,
        generate_kwargs={
            "task": "transcribe",
        },
    )
    predictions_finetuned.append(result["text"])
    references_finetuned.append(example["transcription"])

[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


In [ ]:
finetuned_df = pd.DataFrame({
    "reference": references_finetuned,
    "finetuned_prediction": predictions_finetuned,
})

finetuned_df.to_csv("whisper_finetuned.csv", index=False,)
finetuned_df.head()

,reference,finetuned_prediction
0,And the other is that I was right back in this...,And the others that I was right back in this ...
1,"Right. Like uh, if my parents knew- So our- ou...","Right. Like, um, if my parents knew, so, our-..."
2,"You can look at my son and tell I cook. So, yo...","You can look at my son and tell I cook. So, y..."
3,respectable kids. I would say about thirty per...,"respectable kids, I would say about thirty pe..."
4,"You got a- You got a s- fourteen year old boy,...",You gotta- You gotta s- Fourteen-year-old boy...


In [ ]:
finetuned_wer = wer_metric.compute(
    predictions=predictions_finetuned,
    references=references,
)
print(f"Fine-tuned WER: {finetuned_wer}")

Fine-tuned WER: 0.2516806405317622


## Analysis

## Save Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/whisper_baseline.csv" "/content/drive/MyDrive/asr_project/"
!cp "/content/whisper_finetuned.csv" "/content/drive/MyDrive/asr_project/"

In [ ]:
!cp -r "/content/asr-disparities" "/content/drive/MyDrive/asr_project/"

In [ ]:
!cp -r "/content/whisper-small-fine-tune" "/content/drive/MyDrive/asr_project/"